# 🚀 SEO-AI-MLOps: End-to-End Machine Learning Ranking Prediction Pipeline

This notebook demonstrates the complete supervised machine learning workflow for predicting whether a webpage will achieve a **Google Top-10 SERP ranking** based on technical, on-page, semantic, authority, and intent features.

### Pipeline Overview:
1. **Synthetic & Historical Dataset Generation** (Statistical distributions modeling real-world SERP dynamics)
2. **Exploratory Data Analysis (EDA)** & Feature Correlations
3. **Model Training & Comparison** (XGBoost vs. Random Forest vs. Logistic Regression)
4. **Evaluation Metrics** (ROC-AUC, Precision, Recall, F1, Brier Score, PR Curves)
5. **Global Feature Importance & Local Attribution**

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

from src.models.synthetic_data import SEODatasetGenerator, FEATURE_COLUMNS

sns.set_theme(style="darkgrid")
print("Libraries loaded successfully.")

## 1. Generate & Inspect the SEO Benchmark Dataset

In [ ]:
generator = SEODatasetGenerator(n_samples=3000, random_seed=42)
df = generator.generate()
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis: Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 8))
corr = df[FEATURE_COLUMNS + ['is_top_10']].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation with Google Top-10 Ranking Target", fontsize=14, fontweight="bold")
plt.show()

## 3. Train & Evaluate Models (XGBoost vs. Random Forest vs. Logistic Regression)

In [ ]:
X = df[FEATURE_COLUMNS]
y = df['is_top_10']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Models
lr = LogisticRegression(max_iter=500, random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42).fit(X_train, y_train)
xgb_clf = xgb.XGBClassifier(n_estimators=180, max_depth=5, learning_rate=0.06, random_state=42).fit(X_train, y_train)

lr_probs = lr.predict_proba(X_test)[:, 1]
rf_probs = rf.predict_proba(X_test)[:, 1]
xgb_probs = xgb_clf.predict_proba(X_test)[:, 1]

print(f"Logistic Regression ROC-AUC: {roc_auc_score(y_test, lr_probs):.4f}")
print(f"Random Forest       ROC-AUC: {roc_auc_score(y_test, rf_probs):.4f}")
print(f"XGBoost Classifier  ROC-AUC: {roc_auc_score(y_test, xgb_probs):.4f}")

## 4. Plot Comparison ROC Curves

In [ ]:
plt.figure(figsize=(10, 6))
for name, probs in [('Logistic Regression', lr_probs), ('Random Forest', rf_probs), ('XGBoost', xgb_probs)]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate (Recall)", fontsize=12)
plt.title("ROC Curves: Top-10 SERP Ranking Probability Models", fontsize=14, fontweight="bold")
plt.legend(loc="lower right")
plt.show()

## 5. XGBoost Feature Importance Breakdown

In [ ]:
imp = pd.Series(xgb_clf.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
imp.plot(kind='barh', color='#818CF8')
plt.title("XGBoost Feature Importance in SERP Ranking Outcome", fontsize=14, fontweight="bold")
plt.xlabel("Relative Importance (Gini Gain)")
plt.show()